In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
# BASE = Path("../data/raw")

# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

In [ ]:

avonet_crosswalk = pd.read_csv("../data/raw/Avonet/avonet_crosswalk.csv")

birdbase_crosswalk = pd.read_csv("../data/raw/Birdbase/birdbase_crosswalk.csv")
birdbase = pd.read_csv("../data/processed//birdbase_FE_uncleaned.csv")



In [ ]:
birdbase.isnull().sum()

In [ ]:
null_values = ["null", "NULL", "MISSING", "missing", "Missing", "NA", "na", "N/A", "n/a", ""]

birdbase.replace(null_values, np.nan, inplace=True)

In [ ]:
birdbase.isnull().sum()

In [ ]:
birdbase.head()

In [ ]:
birdbase.columns

In [ ]:
quantitative_feature = [ 'mass_BB', 'elevation_min_BB', 'elevation_range_BB', 
                        'elevation_max_BB']
categorical_feature = ["habitat_BB" ,"primary_diet_BB" , "nest_type_BB"	,"flightless_BB","trophic_niche_BB"]

# before clean

In [ ]:
df_before = birdbase.copy()

In [ ]:
df_before[quantitative_feature].describe()

In [ ]:
df_before.isnull().sum()

## Replace with Median & Mode 
- median = quantitative feature
- mod =  catagorical feature

In [ ]:
df1 = birdbase.copy()

In [ ]:
for col in quantitative_feature:
    if df1[col].isnull().any():
        df1[col] = df1[col].fillna(df1[col].median())

In [ ]:
df1[quantitative_feature].describe()

In [ ]:
for col in categorical_feature:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

In [ ]:
df1.isnull().sum()

## Replace with Mean & Mode 
- mean = quantitative feature
- mod =  catagorical feature

In [ ]:
df2 = birdbase.copy()

In [ ]:
for col in quantitative_feature:
    if df2[col].isnull().any():
        df2[col].fillna(df2[col].mean(), inplace=True)

In [ ]:
df2[quantitative_feature].describe()

In [ ]:
for col in categorical_feature:
    df2[col] = df2[col].fillna(df2[col].mode()[0])

In [ ]:
df2.isnull().sum()

## Mean family/order wise and mod family/order wise

In [ ]:
df3 = birdbase.copy()

In [ ]:
cols = ["avibase_id","family_birdlife", "order_birdlife","family_birdtree","order_birdtree"]

df3 = df3.merge(
    avonet_crosswalk[cols].drop_duplicates("avibase_id"),
    on="avibase_id",
    how="left"
)

In [ ]:
df3 = df3[cols + [c for c in df3.columns if c not in cols]]

In [ ]:
df3.head()

In [ ]:
for col in quantitative_feature:
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdlife")[col].transform("mean")
    )
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdtree")[col].transform("mean")
    )
    # almost 99% covered in above 2 steps
    df3[col] = df3[col].fillna( df3[col].mean() )

In [ ]:
for col in categorical_feature:
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdlife")[col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else None
        )
    )
    
    df3[col] = df3[col].fillna(
        df3.groupby("family_birdtree")[col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else None
        )
    )
    
    # almost 99% covered in above 2 steps
    df3[col] = df3[col].fillna(df3[col].mode().iloc[0])

In [ ]:
df3.isnull().sum()

# Phylogeny 

In [ ]:

# ── 1. Merge birdbase with avonet_crosswalk on avibase_id ──────────────────
merged = birdbase.merge(
    avonet_crosswalk[['avibase_id', 'species_birdtree']],
    on='avibase_id',
    how='left',
    indicator=True
)

# ── 2. Split into matched / unmatched ─────────────────────────────────────
not_in_crosswalk = merged[merged['_merge'] == 'left_only']   # Q1
in_crosswalk     = merged[merged['_merge'] == 'both']        # base for Q2

has_birdtree     = in_crosswalk[in_crosswalk['species_birdtree'].notna()]   # Q2 ✓
no_birdtree      = in_crosswalk[in_crosswalk['species_birdtree'].isna()]    # Q2 ✗

# ── 3. Collect all unmatched avibase_ids ──────────────────────────────────
unmatched_ids = pd.concat([
    not_in_crosswalk[['avibase_id']].assign(reason='not in crosswalk'),
    no_birdtree    [['avibase_id']].assign(reason='in crosswalk, no birdtree_name')
]).reset_index(drop=True)

# ── 4. Summary report ─────────────────────────────────────────────────────
total          = len(birdbase)
q1_missing     = len(not_in_crosswalk)
q2_has_name    = len(has_birdtree)
q2_no_name     = len(no_birdtree)
total_unmatched = len(unmatched_ids)

print("=" * 52)
print(f"  Total avibase_ids in birdbase        : {total:>6}")
print("-" * 52)
print(f"  Q1 — NOT in avonet_crosswalk          : {q1_missing:>6}")
print(f"  Found in avonet_crosswalk             : {total - q1_missing:>6}")
print(f"    ↳ Q2 — WITH birdtree_name           : {q2_has_name:>6}")
print(f"    ↳ Q2 — WITHOUT birdtree_name (null) : {q2_no_name:>6}")
print("-" * 52)
print(f"  Q3 — Total unmatched (Q1 + no name)  : {total_unmatched:>6}")
print("=" * 52)

# ── 5. Inspect unmatched ──────────────────────────────────────────────────
print("\nUnmatched avibase_ids:")
print(unmatched_ids)

In [ ]:
from Bio import Phylo

tree = Phylo.read("../data/processed/Phylogeny.tre", "newick")

tree_species = [term.name for term in tree.get_terminals()]

print(len(tree_species))


In [ ]:
avonet_crosswalk["species_birdtree"] = (
    avonet_crosswalk["species_birdtree"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.capitalize()   # fixes Genus
)

In [ ]:
def precompute_depths(tree):
    depths = {}
    def walk(clade, dist):
        dist += (clade.branch_length or 0.0)
        if clade.is_terminal():
            depths[clade.name] = dist
        for child in clade.clades:
            walk(child, dist)
    walk(tree.root, 0.0)
    return depths


depths = precompute_depths(tree)  

In [ ]:
def get_closest_species(species_name: str, n: int = 25) -> dict:
    path = tree.get_path(species_name)
    target_depth = depths[species_name]
    collected = {}
    cumulative_up = 0.0

    for clade in reversed(path):
        cumulative_up += (clade.branch_length or 0.0)
        for term in clade.get_terminals():
            if term.name != species_name and term.name not in collected:
                collected[term.name] = cumulative_up + (depths[term.name] - (target_depth - cumulative_up))
        if len(collected) >= n:
            break

    return dict(sorted(collected.items(), key=lambda x: x[1])[:n])

In [ ]:
null_values = ["null", "NULL", "MISSING", "missing", "Missing", "NA", "na", "N/A", "n/a", ""]

def fill_null_quant(data, col):
    
    # normalize all null-like values to NaN
    data[col] = data[col].replace(null_values, np.nan)
    
    # loop only over rows where the target column is null
    for idx, row in data[data[col].isnull()].iterrows():
        
        # get the species name for this row
        species = row['species_birdtree']
        
        # if species name itself is null or not in the phylogenetic tree, skip
        if pd.isna(species) or species not in tree_species:
            continue
        
        # get 25 closest species with their distances {name: dist}
        neighbors = get_closest_species(species, n=25)
        
        # from those 25, keep only species that:
        # 1. exist in our dataset's species_birdtree column
        # 2. have a non-null value in the target column
        available = {sp: dist for sp, dist in neighbors.items()
                     if sp in data['species_birdtree'].values and 
                     pd.notna(data.loc[data['species_birdtree'] == sp, col].values[0])}
        
        # if none of the 25 neighbors are in dataset, skip
        if not available:
            continue
        
        # inverse distance weight — closer species get higher weight
        # +1e-9 to avoid division by zero when distance is 0
        weights = {sp: 1/(dist + 1e-9) for sp, dist in available.items()}
        
        # sum of all weights, used for normalization
        total = sum(weights.values())
        
        # normalize so all weights sum to 1
        norm_weights = {sp: w/total for sp, w in weights.items()}
        
        # weighted average — multiply each neighbor's value by its normalized weight and sum
        data.loc[idx, col] = sum(
            norm_weights[sp] * data.loc[data['species_birdtree'] == sp, col].values[0]
            for sp in available
        )
    
    return data


In [ ]:
from collections import defaultdict

def fill_null_cat(data, col):
    
    # normalize all null-like values to NaN
    data[col] = data[col].replace(null_values, np.nan)
    
    # loop only over rows where the target column is null
    for idx, row in data[data[col].isnull()].iterrows():
        
        # get the species name for this row
        species = row['species_birdtree']
        
        # if species name itself is null or not in the phylogenetic tree, skip
        if pd.isna(species) or species not in tree_species:
            continue
        
        # get 25 closest species with their distances {name: dist}
        neighbors = get_closest_species(species, n=50)
        
        # from those 25, keep only species that:
        # 1. exist in our dataset's species_birdtree column
        # 2. have a non-null value in the target column
        available = {sp: dist for sp, dist in neighbors.items()
                     if sp in data['species_birdtree'].values and 
                     pd.notna(data.loc[data['species_birdtree'] == sp, col].values[0])}
        
        # if none of the 25 neighbors are in dataset, skip
        if not available:
            continue
        
        # inverse distance weight — closer species get higher weight
        # +1e-9 to avoid division by zero when distance is 0
        weights = {sp: 1/(dist + 1e-9) for sp, dist in available.items()}
        
        # sum of all weights, used for normalization
        total = sum(weights.values())
        
        # normalize so all weights sum to 1
        norm_weights = {sp: w/total for sp, w in weights.items()}
        
        # sum weights per category instead of weighted average
        category_weights = defaultdict(float)
        for sp in available:
            category = data.loc[data['species_birdtree'] == sp, col].values[0]
            category_weights[category] += norm_weights[sp]
        
        # pick category with highest total weight
        data.loc[idx, col] = max(category_weights, key=category_weights.get)
    
    return data

In [ ]:
df4 = birdbase.copy()

In [ ]:
cols = ["avibase_id","species_birdtree","family_birdlife", "order_birdlife","family_birdtree","order_birdtree"]

df4 = df4.merge(
    avonet_crosswalk[cols].drop_duplicates("avibase_id"),
    on="avibase_id",
    how="left"
)

In [ ]:
df4 = df4[cols + [c for c in df3.columns if c not in cols]]

In [ ]:
null_values = ["null", "NULL", "MISSING", "missing", "Missing", "NA", "na", "N/A", "n/a", ""]

df4.replace(null_values, np.nan, inplace=True)

In [ ]:
df4.isnull().sum()

In [ ]:
for col in quantitative_feature:
    df4 = fill_null_quant(df4, col)

In [ ]:
for col in categorical_feature:
    df4 = fill_null_cat(df4, col)

In [ ]:
df4.isnull().sum()